In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import cv2
from PIL import Image

In [ ]:

mean_gray = 0.1307
std_gray = 0.3081

transforms_ori = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(mean_gray,), std=(std_gray,))
])

transforms_photos = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(mean_gray,), std=(std_gray,))
])

In [ ]:

train_dataset = datasets.MNIST(root="./data", train=True, transform=transforms_ori, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms_ori)
batch_size = 100
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1, stride=1)
        self.batchnorm1 = nn.BatchNorm2d(num_features=8)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=32, kernel_size=5, padding=2, stride=1)
        self.batchnorm2 = nn.BatchNorm2d(num_features=32)
        self.fc1 = nn.Linear(in_features=7 * 7 * 32, out_features=600)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(in_features=600, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.conv2(x)
        x = self.batchnorm2(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = x.view(-1, 7 * 7 * 32)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


model = CNN()

CUDA = torch.cuda.is_available()
if CUDA:
    model = model.cuda()

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

epochs = 10

train_loss = []
test_loss = []
train_accuracy = []
test_accuracy = []

for epoch in range(epochs):
    iter_loss = 0
    correct = 0
    iterations = 0
    model.train()

    for i, (images, labels) in enumerate(train_loader):
        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        output = model(images)
        (_, predicted) = torch.max(output, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(output, labels)
        iter_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        iterations += 1

    train_loss.append(iter_loss / iterations)
    train_accuracy.append((correct / len(train_dataset)) * 100)

    iter_loss = 0
    correct = 0
    iterations = 0
    model.eval()

    for i, (images, labels) in enumerate(test_loader):
        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        output = model(images)
        (_, predicted) = torch.max(output, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(output, labels)
        iter_loss += loss.item()

        iterations += 1

    test_loss.append(iter_loss / iterations)
    test_accuracy.append((correct / len(test_dataset)) * 100)

    print(
        'Epoch {}/{}, Training Loss: {:.3f}, Training Accuracy: {:.3f}, Testing Loss: {:.3f}, Testing Acc: {:.3f}'.format(
            epoch + 1, epochs, train_loss[-1], train_accuracy[-1], test_loss[-1], test_accuracy[-1]))

In [33]:
def predict(img, model):
    img = cv2.imread(img, 0)
    ret, thresholded = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    img = 255 - thresholded
    cv2.imshow("Original", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    img = Image.fromarray(img)
    img = transforms_photos(img)
    img = img.view(1, 1, 28, 28)

    model.eval()

    if CUDA:
        img = img.cuda()
        model = model.cuda()

    output = model(img)
    print(output)
    print(output.data)

    (_, predicted) = torch.max(output, 1)

    return predicted.item()

In [86]:
pred = predict("0.jpg", model)
print(pred)

tensor([[ 1.4717,  0.9299,  1.8629, -0.4901,  0.3252, -0.6284, -1.3305,  0.2740,
         -0.0723, -0.6470]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[ 1.4717,  0.9299,  1.8629, -0.4901,  0.3252, -0.6284, -1.3305,  0.2740,
         -0.0723, -0.6470]], device='cuda:0')
2
